|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Tensor parallelism<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge HELPER: two shardings, one collective<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import torch
torch.manual_seed(0)
HIDDEN, MLP_HIDDEN, RANKS = 256, 1024, 4
inputs = torch.randn(8, HIDDEN)
up_weight = torch.randn(HIDDEN, MLP_HIDDEN) / HIDDEN**0.5
down_weight = torch.randn(MLP_HIDDEN, HIDDEN) / MLP_HIDDEN**0.5
reference = torch.relu(inputs @ up_weight) @ down_weight
print('reference', tuple(reference.shape))

Shard an MLP across four ranks in two different ways. Both ways are correct.
Count what each one costs in conversation.

All of it runs on the CPU. Stage 20 does the same work with real collectives.
This notebook is about the design rule. You must get the rule correct before
the collectives exist.

# Exercise 1: column, then row

In [ ]:
def mlp_parallel(inputs, up_weight, down_weight, ranks):
  """-> (output, number of all-reduces).
  Divide up_weight in one direction and down_weight in the other, so that
  you need one collective. Decide the directions before you write it. relu
  operates on each element, so a rank can complete its hidden units alone.
  """
  up_shards = 
  down_shards = 
  partials = 
  return , 

output, collectives = mlp_parallel(inputs, up_weight, down_weight, RANKS)
print(f'max difference {(output-reference).abs().max().item():.2e}')
print(f'collectives for each block: {collectives}')

# Exercise 2: the other way round

Also correct. Count the collectives.

In [ ]:
def mlp_row_first(inputs, up_weight, down_weight, ranks):
  """Now do it in the other order: row-parallel FIRST.
  Each rank now holds a partial sum of the hidden activations. relu is not
  linear, so you cannot apply it to a partial sum. Count the cost of that."""
  up_shards = list(up_weight.chunk(ranks, dim=0))       # rows: divides the INPUT
  input_shards = list(inputs.chunk(ranks, dim=1))
  hidden = 
  hidden = torch.relu(hidden)
  down_shards = list(down_weight.chunk(ranks, dim=0))
  hidden_shards = list(hidden.chunk(ranks, dim=1))
  output = 
  return output, 

row_first_output, row_first_collectives = mlp_row_first(inputs, up_weight, down_weight, RANKS)
print(f'max difference {(row_first_output-reference).abs().max().item():.2e}   (still correct)')
print(f'collectives for each block: {row_first_collectives}   <- twice the talking, same answer')

# Exercise 3: what a collective costs, with no clock

Each side of this comparison is bytes divided by a bandwidth, so the
milliseconds cancel. Write the ratio of collective time to compute time. You
then find that the layer count also cancels.

Three things remain: your model, your batch, and one number about the machine.
That number is how many times faster the memory is than the interconnect.

In [ ]:
def comm_over_compute(batch, hidden, ranks, bandwidth_ratio, collectives=1):
  """Collective time divided by compute time, with no clock in it.
  A ring all-reduce moves 2(R-1)/R times the tensor. The tensor is B x d
  in bf16. There are L layers and C collectives in each layer. The compute
  side is the weights, W = 24 d^2 L bytes, that R ranks read at HBM speed.
  Write the ratio, and see the layer count and the clock both cancel.
  """
  return 

def break_even_batch(hidden, ranks, bandwidth_ratio, collectives=1):
  """The batch at which the ratio is 1: the ranks talk as long as they compute."""
  return 

BANDWIDTH_RATIO = 25.0     # a PCIe node: HBM is approximately 25x the interconnect
print(f"{'batch':>6} {'1 collective':>14} {'2 collectives':>15}")
for batch in (1, 32, 256):
  one = comm_over_compute(batch, 4096, 8, BANDWIDTH_RATIO, 1)
  two = comm_over_compute(batch, 4096, 8, BANDWIDTH_RATIO, 2)
  print(f'{batch:>6} {one:>14.3f} {two:>15.3f}')
print('\n(the ratio of talking to computing. 1.0 means half your step is the network.)')
# The batch at which talking becomes longer than computing, for three machines.
for bandwidth_ratio, name in ((1.0, 'NVLink'), (25.0, 'PCIe'), (200.0, 'Ethernet')):
  one = 
  two = 
  print(f'{name:>9}: talking becomes longer than computing at batch {one:>7,.0f} '
        f'with 1 collective, {two:>7,.0f} with 2')

### Before you open the solution

1. Both versions give the right answer. Why does the second one need two
   collectives? Which operation is in the way?
2. State the rule in one sentence, so that it also tells you how to
   split attention.
3. Look at your table at batch 256. If you are on PCIe, what is the
   largest batch you can run before the collectives cost more than the
   step? What does that do to the throughput argument from Part 2?